In [1]:
pip install torch numpy tslearn pandas mantis-tsfm


Note: you may need to restart the kernel to use updated packages.


# Project DL4TS
### By Alice Lataste, Thomas Roussaux, Spyridon Paipetis and Soline Mignot

The project aims to do some time series classification on the LSST dataset from the tslearn library.

In [2]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score
#from tslearn.shapelets import LearningShapelets
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tslearn.datasets import UCR_UEA_datasets
import pandas as pd
import numpy as np
import gc

In [3]:
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

Device: cuda


## Fetching the dataset and normalizing it

In [4]:
from tslearn.datasets import UCR_UEA_datasets
# Load the LSST dataset from UEA archive
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

In [5]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(len(set(y_train)))

(2459, 36, 6)
(2466, 36, 6)
(2459,)
14


In [6]:
# Normalization
scaler = TimeSeriesScalerMeanVariance()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# encode labels
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# conversion torch
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

## Part 1 : Baseline model: CNN

### Step 1 : creating the class

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNBaseline(nn.Module):
    def __init__(self, n_channels=6, n_classes=14):
        super().__init__()
        
        self.conv1 = nn.Conv1d(in_channels=n_channels, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        self.fc1 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, n_classes)

    def forward(self, x):
        # x: (batch, 36, 6) -> (batch, 6, 36)
        x = x.permute(0, 2, 1)
        
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        
        x = self.pool(x).squeeze(-1)   # (batch, 128)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

### Step 2 : Model initialization and training


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = CNNBaseline(n_channels=6, n_classes=14).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

cuda


In [9]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * X_batch.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
    
    return total_loss / total, correct / total

### Step 3 : Evaluation

In [10]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            total_loss += loss.item() * X_batch.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    
    return total_loss / total, correct / total

### Step 4 : Run the model

In [11]:
n_epochs = 100

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    print(f"Epoch {epoch+1:02d}/{n_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Epoch 01/100 | Train Loss: 2.5115 | Train Acc: 0.2513 | Test Loss: 2.3728 | Test Acc: 0.3151
Epoch 02/100 | Train Loss: 2.2538 | Train Acc: 0.3160 | Test Loss: 2.1402 | Test Acc: 0.3151
Epoch 03/100 | Train Loss: 2.0876 | Train Acc: 0.3213 | Test Loss: 1.9989 | Test Acc: 0.3151
Epoch 04/100 | Train Loss: 1.9785 | Train Acc: 0.3444 | Test Loss: 1.8971 | Test Acc: 0.3755
Epoch 05/100 | Train Loss: 1.8815 | Train Acc: 0.3977 | Test Loss: 1.8085 | Test Acc: 0.4720
Epoch 06/100 | Train Loss: 1.8121 | Train Acc: 0.4282 | Test Loss: 1.7345 | Test Acc: 0.4785
Epoch 07/100 | Train Loss: 1.7554 | Train Acc: 0.4514 | Test Loss: 1.6713 | Test Acc: 0.5203
Epoch 08/100 | Train Loss: 1.6916 | Train Acc: 0.4835 | Test Loss: 1.6156 | Test Acc: 0.5260
Epoch 09/100 | Train Loss: 1.6466 | Train Acc: 0.4860 | Test Loss: 1.5655 | Test Acc: 0.5430
Epoch 10/100 | Train Loss: 1.5935 | Train Acc: 0.5136 | Test Loss: 1.5152 | Test Acc: 0.5547
Epoch 11/100 | Train Loss: 1.5540 | Train Acc: 0.5230 | Test Loss: 1.4

## Part 2 : Adapting a foundational model : Mantis

For this second part, we have decided to do Setting 1 where we adapt a foundational model. We have chosen to adapt the Mantis foundational model.

In [12]:
from mantis.architecture import MantisV1

network = MantisV1(device=device)
network = network.from_pretrained("paris-noah/Mantis-8M")


/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:

from tslearn.datasets import UCR_UEA_datasets
import numpy as np
import torch
import torch.nn.functional as F
from mantis.architecture import Mantis8M
from mantis.trainer import MantisTrainer
from mantis.adapters import LinearChannelCombiner
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Load data
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")
X_train = X_train.astype('float32')  
X_test  = X_test.astype('float32')
y_train = y_train.astype('float32')
y_test = y_test.astype('float32')


le = LabelEncoder()
y_train = le.fit_transform(y_train) 
y_test  = le.transform(y_test)

def resize(X, resize_size):
    X_t = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1) 
    X_resized = F.interpolate(X_t, size=resize_size, mode='linear', align_corners=False)  
    return X_resized.numpy() 

X_train_resized = resize(X_train, 512)  
X_test_resized  = resize(X_test, 512)  

# Confirm shapes
print(X_train_resized.shape)  # should be (2459, 6, 512)
print(y_train.shape, y_train.dtype)



(2459, 6, 512)
(2459,) int64


### Step 1 : Choose if we adapt the full model or just the head

The first step in adapting the foundational models, is to choose whether we will be re-adapting the full model or adapter and head. Also, we need to choose the right amount of epochs for this program to run. 

The most common issue is to either not train enough, and not have a strong enough model or to train too much and to overfit the model.
That is especially the risk when adapting the full model.

To proceed, we are going do a list : amount_epochs = [5 , 10, 15, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100, 150] , and have a loop that trains the model for these amount of epochs and that keeps track of the performances of the model. 

We can then study the performances on the test and train datasets to see which epoch amount is best.

First, we see that best amount of epochs when adapting just the adapter and the head: 

In [14]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report

def find_best_epoch_number_for_adapter_head(amount_epochs):
    network = Mantis8M(device=device)
    network = network.from_pretrained("paris-noah/Mantis-8M")

    rows = []

    for numb_epoch in amount_epochs : 
        print(f"\nCurrently doing {numb_epoch} epochs: ")
        # num_channels=X.shape[1] → 6 channels (C dimension), matching official docs
        adapter = LinearChannelCombiner(num_channels=X_train_resized.shape[1], new_num_channels=5)
        model = MantisTrainer(device=device, network=network)
        model.fit(X_train_resized, y_train, adapter=adapter, fine_tuning_type='adapter_head', 
            num_epochs=numb_epoch)

        y_pred_train = model.predict(X_train_resized)
        y_pred_test  = model.predict(X_test_resized)
        
        row = {
            'num_epochs':      numb_epoch,

            # Accuracy
            'train_acc':       accuracy_score(y_train, y_pred_train),
            'test_acc':        accuracy_score(y_test,  y_pred_test),

            # Balanced accuracy (good for imbalanced LSST classes)
            'train_bal_acc':   balanced_accuracy_score(y_train, y_pred_train),
            'test_bal_acc':    balanced_accuracy_score(y_test,  y_pred_test),

            # Macro F1 (penalises poor performance on minority classes)
            'train_f1_macro':  f1_score(y_train, y_pred_train, average='macro'),
            'test_f1_macro':   f1_score(y_test,  y_pred_test,  average='macro'),

            # Weighted F1 (accounts for class frequency)
            'train_f1_weighted': f1_score(y_train, y_pred_train, average='weighted'),
            'test_f1_weighted':  f1_score(y_test,  y_pred_test,  average='weighted'),

            # Overfit gap — if this grows, you're overfitting
            'overfit_gap':     round(accuracy_score(y_train, y_pred_train) - 
                                    accuracy_score(y_test,  y_pred_test), 4),
        }
        rows.append(row)

        print(f"[{numb_epoch:>3} epochs ] "
            f"test_acc={row['test_acc']:.4f} | "
            f"test_bal={row['test_bal_acc']:.4f} | "
            f"test_f1={row['test_f1_macro']:.4f} | "
            f"gap={row['overfit_gap']:+.4f}")

    accuracy_per_epoch = pd.DataFrame(rows)
    return accuracy_per_epoch

amount_epochs = [3, 5, 10, 15, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100, 150] 
#accuracy_per_epoch = find_best_epoch_number_for_adapter_head(amount_epochs)
#accuracy_per_epoch.to_csv('accuracy_per_epoch.csv')

"""
This function does a lot of training and thus takes a long time. 
We have saved the results that we are now printing, in order to pick the best.
"""

accuracy_per_epoch = pd.read_csv('accuracy_per_epoch.csv', index_col=0)

print(accuracy_per_epoch)



    num_epochs  train_acc  test_acc  train_bal_acc  test_bal_acc  \
0            3   0.128914  0.124088       0.082825      0.088558   
1            5   0.361529  0.366586       0.147592      0.146873   
2           10   0.489223  0.475669       0.250038      0.245834   
3           15   0.557137  0.534874       0.304637      0.286905   
4           20   0.555104  0.530008       0.311258      0.289852   
5           25   0.621391  0.580292       0.373567      0.345388   
6           30   0.633184  0.596918       0.428196      0.366585   
7           40   0.674664  0.616383       0.481526      0.407787   
8           50   0.678325  0.622060       0.509864      0.396856   
9           60   0.692558  0.631387       0.515531      0.436226   
10          70   0.718178  0.634631       0.557651      0.437689   
11          80   0.753152  0.655312       0.623519      0.485704   
12          90   0.738512  0.632198       0.636817      0.448751   
13         100   0.736072  0.622871       0.6250

For this adapter head, the best results are observed at 80 epochs, where the test balanced accuracy peaks at 0.486 and the test F1 macro score reaches 0.506. Beyond this point, the model shows signs of plateauing or even slight degradation in test performance, despite continued improvement in training metrics. The overfit gap also widens significantly after 80 epochs, reaching 0.1978 at 200 epochs, indicating overfitting to the training data. This suggests that 80 epochs strikes the best balance between training and generalization for this configuration.

Timing-wise, it takes 12.57 s /epoch. So an 80 epoch training takes just under 17 minutes, which can be quite long. 

Let's see the performances when adapting the full model.

In [15]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report

def find_best_epoch_number_for_full(amount_epochs):
    network = Mantis8M(device=device)
    network = network.from_pretrained("paris-noah/Mantis-8M")

    amount_epochs = [15, 20, 25, 35]
    full_rows = []

    for numb_epoch in amount_epochs : 
        print(f"\nCurrently doing {numb_epoch} epochs: ")
        # num_channels=X.shape[1] → 6 channels (C dimension), matching official docs
        adapter = LinearChannelCombiner(num_channels=X_train_resized.shape[1], new_num_channels=5)
        model = MantisTrainer(device='cuda', network=network)
        model.fit(X_train_resized, y_train, adapter=adapter, fine_tuning_type='full', 
            num_epochs=numb_epoch)

        y_pred_train = model.predict(X_train_resized)
        y_pred_test  = model.predict(X_test_resized)
        
        row = {
            'num_epochs':      numb_epoch,

            # Accuracy
            'train_acc':       accuracy_score(y_train, y_pred_train),
            'test_acc':        accuracy_score(y_test,  y_pred_test),

            # Balanced accuracy (good for imbalanced LSST classes)
            'train_bal_acc':   balanced_accuracy_score(y_train, y_pred_train),
            'test_bal_acc':    balanced_accuracy_score(y_test,  y_pred_test),

            # Macro F1 (penalises poor performance on minority classes)
            'train_f1_macro':  f1_score(y_train, y_pred_train, average='macro'),
            'test_f1_macro':   f1_score(y_test,  y_pred_test,  average='macro'),

            # Weighted F1 (accounts for class frequency)
            'train_f1_weighted': f1_score(y_train, y_pred_train, average='weighted'),
            'test_f1_weighted':  f1_score(y_test,  y_pred_test,  average='weighted'),

            # Overfit gap — if this grows, you're overfitting
            'overfit_gap':     round(accuracy_score(y_train, y_pred_train) - 
                                    accuracy_score(y_test,  y_pred_test), 4),
        }
        full_rows.append(row)

        print(f"[{numb_epoch:>3} epochs ] "
            f"test_acc={row['test_acc']:.4f} | "
            f"test_bal={row['test_bal_acc']:.4f} | "
            f"test_f1={row['test_f1_macro']:.4f} | "
            f"gap={row['overfit_gap']:+.4f}")

        pd.DataFrame(full_rows).to_csv('mantis_full_results.csv', index=False)

    full_accuracy_per_epoch = pd.DataFrame(full_rows)
    return full_accuracy_per_epoch

amount_epochs = [5, 10, 15, 20, 25, 30, 35, 40, 50, 60] 
#full_accuracy_per_epoch = find_best_epoch_number_for_full(amount_epochs)
#full_accuracy_per_epoch.to_csv('full_accuracy_per_epoch.csv')

"""
This function does a lot of training and thus takes a long time. 
We have saved the results that we are now printing, in order to pick the best.
"""

full_accuracy_per_epoch = pd.read_csv('full_accuracy_per_epoch.csv', index_col=0)

print(full_accuracy_per_epoch)


   num_epochs  train_acc  test_acc  train_bal_acc  test_bal_acc  \
0          10   0.695811  0.623277       0.474221      0.391307   
1          15   0.785685  0.644769       0.641524      0.467215   
2          20   0.902399  0.653285       0.823100      0.486482   
3          25   0.969906  0.656934       0.933389      0.505051   
4          30   0.988613  0.650852       0.971741      0.481739   
5          35   0.994713  0.548256       0.998573      0.437408   
6          40   0.998780  0.630576       0.999004      0.491674   
7          50   1.000000  0.619627       1.000000      0.490569   
8          60   1.000000  0.592052       1.000000      0.464859   

   train_f1_macro  test_f1_macro  train_f1_weighted  test_f1_weighted  \
0        0.503939       0.390355           0.667555          0.590798   
1        0.673881       0.464892           0.753295          0.602422   
2        0.864007       0.500722           0.898424          0.633035   
3        0.954585       0.524311     

For this adapter head, 20 epochs represents a strong balance between training efficiency and test performance. At this point, the model achieves a test balanced accuracy of 0.486 and a test F1 macro score of 0.501, both of which are near their peak values. The training metrics are already very high (e.g., train balanced accuracy at 0.823), but the test performance remains competitive without the severe overfitting seen in later epochs. The overfit gap at 20 epochs is 0.249, which is more controlled compared to the wider gaps observed beyond 25 epochs.

While 25 epochs slightly outperforms in absolute test metrics, 20 epochs still delivers robust results with a lower risk of overfitting, making it a practical and efficient choice for this configuration.

Timing-wise, it takes 16.81s /epoch. So an 20 epoch training takes just under 6 minutes, which is makes faster than when changing adapter_head, while having the same performances.

Therefore, we now choose to adapt the full model, with 20 epochs. Let's now find the best parameters.

### Step 2 : Fine-tune the parameters 

In this next part, we are going to fine-tune the parameters. We are exploring : normalization, class weighting, learning rate, and adapter channel count on model performance. By testing these parameters in isolation and combination, we aim to identify the optimal configuration for balancing accuracy, robustness, and training efficiency.

In [18]:
from mantis.architecture import Mantis8M
from mantis.trainer import MantisTrainer
from mantis.adapters import LinearChannelCombiner
import torch.nn.functional as F

def best_test_size():
    ds = UCR_UEA_datasets()
    X_train_raw, y_train, X_test_raw, y_test = ds.load_dataset('LSST')

    X_train_raw = X_train_raw.astype('float32')  
    X_test_raw  = X_test_raw.astype('float32')
    y_train = y_train.astype('float32')
    y_test = y_test.astype('float32')

    X_train_raw = np.nan_to_num(X_train_raw, nan=0.0).transpose(0, 2, 1)
    X_test_raw  = np.nan_to_num(X_test_raw,  nan=0.0).transpose(0, 2, 1)
        

    def resize(X, size=128):
        X_tensor = torch.tensor(X, dtype=torch.float)
        return F.interpolate(X_tensor, size=size, mode='linear', align_corners=False).numpy()

    sizes = [128, 256, 512]
    results = []

    for size in sizes:
        print(f"\n--- Testing size: {size} ---")
        X_train_resized = resize(X_train_raw, size=size)
        X_test_resized  = resize(X_test_raw,  size=size)

        network = Mantis8M(device='cuda')
        network = network.from_pretrained('paris-noah/Mantis-8M')

        adapter = LinearChannelCombiner(num_channels=X_train_resized.shape[1], new_num_channels=5)
        model = MantisTrainer(device='cuda', network=network)
        model.fit(X_train_resized, y_train, adapter=adapter,
                  num_epochs=20, fine_tuning_type='full')

        y_pred_train = model.predict(X_train_resized)
        y_pred_test  = model.predict(X_test_resized)

        row = {
            'size':              size,
            # Test metrics
            'test_acc':          accuracy_score(y_test,  y_pred_test),
            'test_bal_acc':      balanced_accuracy_score(y_test,  y_pred_test),
            # Train metrics
            'train_acc':         accuracy_score(y_train, y_pred_train),
            'train_bal_acc':     balanced_accuracy_score(y_train, y_pred_train),
            # Overfit gap
            'overfit_gap':       round(accuracy_score(y_train, y_pred_train) -
                                       accuracy_score(y_test,  y_pred_test), 4),
        }
        results.append(row)
        pd.DataFrame(results).to_csv('mantis_size_experiments.csv', index=False)
        print(f"  test_acc={row['test_acc']:.4f} | test_bal={row['test_bal_acc']:.4f} | "
              f"gap={row['overfit_gap']:+.4f}")

    results_df = pd.DataFrame(results)
    results_df.to_csv('mantis_size_experiments.csv', index=False)
    return results_df

results_df = best_test_size()
results_df = pd.read_csv('mantis_size_experiments.csv')
print(results_df.sort_values('test_acc', ascending=False))



--- Testing size: 128 ---


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler

def run_parameter_experiments(experiments, file_name = 'mantis_experiments.csv'):
    ds = UCR_UEA_datasets()
    X_train_raw, y_train_raw, X_test_raw, y_test_raw = ds.load_dataset('LSST')
    
    def resize(X, size=128):
        X_tensor = torch.tensor(X, dtype=torch.float)
        return F.interpolate(X_tensor, size=size, mode='linear', align_corners=False).numpy()

    # Preprocessing: NaN -> 0, transpose
    X_train_raw = np.nan_to_num(X_train_raw, nan=0.0).transpose(0, 2, 1)  # (2459, 6, 36)
    X_test_raw = np.nan_to_num(X_test_raw, nan=0.0).transpose(0, 2, 1)    # (2466, 6, 36)

    X_train_resized = resize(X_train_raw, size=512)
    X_test_resized = resize(X_test_raw, size=512)

    """
    Runs a series of experiments to fine-tune normalization, class weights, learning rate, and channel count.
    Saves results to 'mantis_experiments.csv' for analysis.
    """
    network = Mantis8M(device=device)
    network = network.from_pretrained("paris-noah/Mantis-8M")

    exp_rows = []

    for exp in experiments:
        print(f"\n--- Running: {exp['name']} ---")

        # Normalization
        if exp['normalize']:
            X_tr = X_train_resized.copy()
            X_te = X_test_resized.copy()
            for c in range(X_tr.shape[1]):
                scaler = StandardScaler()
                X_tr[:, c, :] = scaler.fit_transform(X_tr[:, c, :])
                X_te[:, c, :] = scaler.transform(X_te[:, c, :])
        else:
            X_tr, X_te = X_train_resized, X_test_resized

        # Class weights
        if exp['class_weights']:
            cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
            criterion = torch.nn.CrossEntropyLoss(
                weight=torch.tensor(cw, dtype=torch.float32).to(device))
        else:
            criterion = torch.nn.CrossEntropyLoss()

        # Adapter and training
        adapter = LinearChannelCombiner(num_channels=X_tr.shape[1], new_num_channels=exp['new_ch'])
        model = MantisTrainer(device=device, network=network)
        model.fit(X_tr, y_train,
                  adapter=adapter,
                  fine_tuning_type='full',
                  num_epochs=15,
                  base_learning_rate=exp['lr'],
                  criterion=criterion)

        y_pred_train = model.predict(X_tr)
        y_pred_test  = model.predict(X_te)

        row = {
            'name': exp['name'],
            'test_acc': accuracy_score(y_test, y_pred_test),
            'test_bal_acc': balanced_accuracy_score(y_test, y_pred_test),
            'test_f1_macro': f1_score(y_test, y_pred_test, average='macro'),
            'test_f1_weighted': f1_score(y_test, y_pred_test, average='weighted'),
            'overfit_gap': round(accuracy_score(y_train, y_pred_train) - accuracy_score(y_test, y_pred_test), 4),
        }
        exp_rows.append(row)
        pd.DataFrame(exp_rows).to_csv(file_name, index=False)

        print(f"  test_acc={row['test_acc']:.4f} | bal={row['test_bal_acc']:.4f} | "
              f"f1={row['test_f1_macro']:.4f} | gap={row['overfit_gap']:+.4f}")

    exp_df = pd.DataFrame(exp_rows)
    return exp_df

# Uncomment to run experiments
experiments = [
    # Baseline (for reference)
    #{'name': 'baseline',          'lr': 1e-4, 'new_ch': 5, 'normalize': False, 'class_weights': False},
    {'name': 'lr_1e-3',        'lr': 1e-3, 'new_ch': 5, 'normalize': False, 'class_weights': False},
    {'name': 'normalized',       'lr': 1e-3, 'new_ch': 5, 'normalize': True, 'class_weights': False},
    {'name': 'class_weights',    'lr': 1e-3, 'new_ch': 5, 'normalize': False, 'class_weights': True},
    {'name': 'norm+cw',          'lr': 1e-3, 'new_ch': 5, 'normalize': True, 'class_weights': True},

    # higher LR (best so far)
    {'name': 'lr_5e-4',        'lr': 5e-4, 'new_ch': 5, 'normalize': False, 'class_weights': False},
    {'name': 'lr_2e-4',        'lr': 2e-4, 'new_ch': 5, 'normalize': False, 'class_weights': False},

    # Channel variants with best LR (1e-3)
    {'name': 'cw_ch_3_lr_1e-3',   'lr': 1e-3, 'new_ch': 3, 'normalize': False, 'class_weights': False},
    {'name': 'cw_ch_4_lr_1e-3',   'lr': 1e-3, 'new_ch': 4, 'normalize': False, 'class_weights': False},
    {'name': 'cw_ch_6_lr_1e-3',   'lr': 1e-3, 'new_ch': 6, 'normalize': False, 'class_weights': False},

    # Optional: Test intermediate LR (2e-4)
]


exp_df = run_parameter_experiments(experiments, file_name = 'mantis_experiments.csv')
exp_df.to_csv('mantis_experiments.csv', index=False)

# Load saved results instead
exp_df = pd.read_csv('mantis_experiments.csv', index_col=0)
print(exp_df.sort_values('test_bal_acc', ascending=False))


In [ ]:
def run_parameter_experiments(experiments, file_name = 'mantis_experiments.csv'):
    ds = UCR_UEA_datasets()
    X_train_raw, y_train_raw, X_test_raw, y_test_raw = ds.load_dataset('LSST')
    
    def resize(X, size=128):
        X_tensor = torch.tensor(X, dtype=torch.float)
        return F.interpolate(X_tensor, size=size, mode='linear', align_corners=False).numpy()

    # Preprocessing: NaN -> 0, transpose
    X_train_raw = np.nan_to_num(X_train_raw, nan=0.0).transpose(0, 2, 1)  # (2459, 6, 36)
    X_test_raw = np.nan_to_num(X_test_raw, nan=0.0).transpose(0, 2, 1)    # (2466, 6, 36)

    X_train_resized = resize(X_train_raw, size=512)
    X_test_resized = resize(X_test_raw, size=512)

    """
    Runs a series of experiments to fine-tune normalization, class weights, learning rate, and channel count.
    Saves results to 'mantis_experiments.csv' for analysis.
    """
    network = Mantis8M(device=device)
    network = network.from_pretrained("paris-noah/Mantis-8M")

    exp_rows = []

    for exp in experiments:
        print(f"\n--- Running: {exp['name']} ---")

        # Normalization
        if exp['normalize']:
            X_tr = X_train_resized.copy()
            X_te = X_test_resized.copy()
            for c in range(X_tr.shape[1]):
                scaler = StandardScaler()
                X_tr[:, c, :] = scaler.fit_transform(X_tr[:, c, :])
                X_te[:, c, :] = scaler.transform(X_te[:, c, :])
        else:
            X_tr, X_te = X_train_resized, X_test_resized

        # Class weights
        if exp['class_weights']:
            cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
            criterion = torch.nn.CrossEntropyLoss(
                weight=torch.tensor(cw, dtype=torch.float32).to(device))
        else:
            criterion = torch.nn.CrossEntropyLoss()

        # Adapter and training
        adapter = LinearChannelCombiner(num_channels=X_tr.shape[1], new_num_channels=exp['new_ch'])
        model = MantisTrainer(device=device, network=network)
        model.fit(X_tr, y_train,
                  adapter=adapter,
                  fine_tuning_type='full',
                  num_epochs=20,
                  base_learning_rate=exp['lr'],
                  criterion=criterion)

        y_pred_train = model.predict(X_tr)
        y_pred_test  = model.predict(X_te)

        row = {
            'name': exp['name'],
            'test_acc': accuracy_score(y_test, y_pred_test),
            'test_bal_acc': balanced_accuracy_score(y_test, y_pred_test),
            'test_f1_macro': f1_score(y_test, y_pred_test, average='macro'),
            'test_f1_weighted': f1_score(y_test, y_pred_test, average='weighted'),
            'overfit_gap': round(accuracy_score(y_train, y_pred_train) - accuracy_score(y_test, y_pred_test), 4),
        }
        exp_rows.append(row)
        pd.DataFrame(exp_rows).to_csv(file_name, index=False)

        print(f"  test_acc={row['test_acc']:.4f} | bal={row['test_bal_acc']:.4f} | "
              f"f1={row['test_f1_macro']:.4f} | gap={row['overfit_gap']:+.4f}")

    exp_df = pd.DataFrame(exp_rows)
    return exp_df

In [ ]:
new_experiments = [
    {'name': 'lr_5e-5',        'lr': 5e-5, 'new_ch': 5, 'normalize': False, 'class_weights': False},
    {'name': 'lr_1e-5',        'lr': 1e-5, 'new_ch': 5, 'normalize': False, 'class_weights': False},
    {'name': 'lr_5e-3',        'lr': 5e-3, 'new_ch': 5, 'normalize': False, 'class_weights': False},
    {'name': 'lr_1e-2',        'lr': 1e-2, 'new_ch': 5, 'normalize': False, 'class_weights': False},
    {'name': 'lr_5e-2',        'lr': 5e-2, 'new_ch': 5, 'normalize': False, 'class_weights': False},
]

exp_df = run_parameter_experiments(new_experiments, file_name = 'mantis_new_experiments.csv')
exp_df.to_csv('mantis_new_experiments.csv', index=False)

# Load saved results instead
exp_df = pd.read_csv('mantis_new_experiments.csv', index_col=0)
print(exp_df.sort_values('test_bal_acc', ascending=False))
